# Acoustic Scan 

In [1]:
# matching pursuit
# depth profiling
# attenuation with high f. reflection ok, transmission no
# look at acoustic resonances, dip in attenuation
# 

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
from matplotlib import pyplot as plt
import sys

sys.path.append('..') # path to the src directory
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/ultrasonicTesting')
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/M3Learning-Util/src')
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/AutoPhysLearn/src')
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/Gaussian_Sampler/Gaussian_Sampler')


from scipy.signal import butter, sosfiltfilt
import copy
import math
import time
from tqdm import tqdm
import pickleJar as pj
import tomography as tm

In [2]:
from viz.visualize_scan_data import *
from IPython.display import display
import plotly.graph_objects as go

## Dataloader with preprocessing

In [3]:
from Gaussian_Sampler.data import datasets
from Gaussian_Sampler.data.datasets import morlet_1D_dataset_real

dset = morlet_1D_dataset_real(sq3lite_path='/home/xinqiao/new_mount/gaussian_sampler/ultrasound_data/SA_tomography_water_realigned.sqlite3',
                              dset_name='voltage_transmission_forward',
                              image_shape = (1,1))

sqliteToPickle Warning: pickle file /home/xinqiao/new_mount/gaussian_sampler/ultrasound_data/SA_tomography_water_realigned.pickle already exists. Conversion aborted.


/home/xinqiao/new_mount/gaussian_sampler/ultrasonicTesting/pickleJar.py:1185: RuntimeWarning: divide by zero encountered in log10
  logData = np.log10(abs(data))


In [4]:
# dset.display_dict_tree()

## Interactive Viewer with Slider

Use the slider below to browse through all scans interactively.

In [5]:
# # Create interactive viewer with slider (fast - uses ipywidgets)
# from Gaussian_Sampler.viz.visualize_scan_data import plotly_viewer
# viewer = plotly_viewer(dset)
# display(viewer)  # or just: viewer  (in Jupyter, the last line auto-displays)

## try training model on water with morlet packet

goals:
- figure out mean position and f of morlet packet
- using this, calculate speed of sound in this water

In [6]:
from Gaussian_Sampler.models.morlet_fitter import Fitter_AE, morlet_1D_fitters_real
from autophyslearn.spectroscopic.nn import block_factory, Conv_Block, FC_Block  # pyright: ignore[reportMissingImports]
from autophyslearn.spectroscopic.nn import Multiscale1DFitter
from Gaussian_Sampler.data.custom_sampler import Gaussian_Sampler
import torch

num_fits = 8 # number of curves to sum up
num_params = 4 # number of parameters to fit
# todo: change wandb naming to include noise level, group and regularization technique
# todo: test more num fits
model = Fitter_AE(function=morlet_1D_fitters_real,
                dset=dset,
                num_params=num_params,
                num_fits=num_fits,
                checkpoints_label='ultrasound_water',
                input_channels = 1,
                learning_rate=3e-6,
                device='cuda:0',
                encoder = Multiscale1DFitter,
                encoder_params = {
                    "model_block_dict": { # factory wrapper for blocks
                            "hidden_x1": block_factory(Conv_Block)(output_channels_list=[256,128], 
                                                                    kernel_size_list=[5,5], 
                                                                    pool_list=[10000,500], 
                                                                    max_pool=False),
                            "hidden_xfc": block_factory(FC_Block)(output_size_list=[128,64]), # remove 2nd block and skip connections
                            "hidden_x2": block_factory(Conv_Block)(output_channels_list=[32,16], 
                                                                    kernel_size_list=[3,3], 
                                                                    pool_list=[64,32], 
                                                                    max_pool=True),
                            "hidden_embedding": block_factory(FC_Block)(output_size_list=[8*num_fits,num_params*num_fits], last=True),
                        },
                        # TEST: LIMITS,
                        # "skip_connections": {'hidden_xfc': 'hidden_embedding'},
                        "skip_connections": {},
                        "function_kwargs": {'limits': [1, # amplitude
                                                       dset.spec_len, # mean
                                                       dset.spec_len/10, # stdev
                                                       1/dset.spec_len*100] # freq
                                            } 
                    },
                    # sampler = Gaussian_Sampler, # using random sampler
                    # sampler_params = {'dset': dset, 
                    #                     'batch_size': 100, 
                    #                     'gaussian_std': 3, 
                    #                     'orig_shape': dset.shape[0:-1], 
                    #                     'num_neighbors': 10, },
                )


/home/xinqiao/anaconda3/envs/gaussian_sampler/lib/python3.13/site-packages/datafed_torchflow/computer.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### make graph for model


In [ ]:
# nn.Tanh()

### Train model for several epochs


In [8]:
# import wandb
# wandb.init(group='sub_sampler_type', name='sub_noise_level') # later change config for regularization
model.train(epochs=51,save_every=50, log_wandb=False)

/home/xinqiao/new_mount/gaussian_sampler/ultrasound_data/ultrasound_water/checkpoints/voltage_transmission_forward


100%|██████████| 1/1 [00:00<00:00,  2.32it/s]


Epoch: 000/051 | Train Loss: 0.1067
.............................


100%|██████████| 1/1 [00:00<00:00, 95.65it/s]


Epoch: 001/051 | Train Loss: 0.0673
.............................


...

Epoch: 048/051 | Train Loss: 0.0071
.............................


100%|██████████| 1/1 [00:00<00:00, 127.95it/s]


Epoch: 049/051 | Train Loss: 0.0070
.............................


100%|██████████| 1/1 [00:00<00:00, 118.46it/s]


Epoch: 050/051 | Train Loss: 0.0069
.............................


### Embeddings

In [9]:
def write_scaled_embedding(batch_size=1):
    for i, (idx, x) in enumerate(tqdm(model.dataloader, leave=True, total=len(model.dataloader))):
        with torch.no_grad():
            fits, params = model.encoder(x.float().to(model.device))
            fits = fits.cpu().numpy()
            params = params.cpu().numpy()
    return fits, params

fits, params = write_scaled_embedding(batch_size=1)

# sweep frequencies and search for resonances 
# attenuation in each layer accounts for spherical nature of wave
# data for one to 20 layers
# measure waveform from 30-50, measuring thermal gradient. shoul dbe the same as 40 (mean), so why isnt it?
# goal: use ultrasound to monitor the thermal expansion so we can decreasing charging rate. this way the battery is less likely to experience stress and can cycle more

100%|██████████| 1/1 [00:00<00:00, 160.09it/s]


In [10]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib

viridis = matplotlib.colormaps.get_cmap('viridis').resampled(fits.shape[1])

x, y = 0, 0
idx = y * dset.shape[0] + x

# Build parameter table data
param_table = {'channel': [], 'a': [], 'mu': [], 'sigma': [], 'omega': []}
for i in range(fits.shape[1]):
    param_table['channel'].append(i)
    param_table['a'].append(f'{params[idx][i][0]:.2f}')
    param_table['mu'].append(f'{params[idx][i][1]:.0f}')
    param_table['sigma'].append(f'{params[idx][i][2]:.0f}')
    param_table['omega'].append(f'{params[idx][i][3]:.2e}')

# Highlight rows where a != 0
cell_colors = ['#ffffb3' if float(a) != 0 else 'white' for a in param_table['a']]
# Build table_fill_colors per Plotly spec: a list of lists of shape [num_columns][num_rows].
table_fill_colors = [cell_colors]*5

# --- Font colors section (must be shape [num_rows][num_columns] for Plotly) ---
font_colors = [matplotlib.colors.rgb2hex(viridis(i)) for i in param_table['channel']]
table_font_colors = [font_colors]*5

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{"type": "scatter"}, {"type": "table"}]]
)

# --- Plot traces ---
fig.add_trace(go.Scatter(x=list(range(len(dset[idx][1]))), y=dset[idx][1],
                         mode='lines', name='original'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(fits.shape[2])), y=fits[idx].sum(axis=0),
                         mode='lines', name='sum'), row=1, col=1)
for i in range(fits.shape[1]):
    color = matplotlib.colors.rgb2hex(viridis(i))
    fig.add_trace(go.Scatter(
        x=list(range(fits.shape[2])), y=fits[idx][i],
        mode='lines', 
        line=dict(dash='dot', width=1, color=color),
        showlegend=False
    ), row=1, col=1)

# --- Table ---
fig.add_trace(go.Table(
    header=dict(values=['channel','a', 'mu', 'sigma', 'omega'],
                fill_color='#4a4a4a',
                font=dict(color='white'),
                align='center'),
    cells=dict(values=[param_table['channel'],
                      param_table['a'], 
                      param_table['mu'],
                      param_table['sigma'], 
                      param_table['omega']],
               fill_color=table_fill_colors,
               font=dict(color=table_font_colors),  # shape: [rows][columns]
               align='center')
), row=1, col=2)

fig.update_layout(title='Comparing original and fitted Morlet packet (a, mu, sigma, omega)')
fig.show()